# Módulo 04 — Avaliação de Modelos
**Portfólio de Laboratório de RNA**
**Aluno:** Gabriel Rocha Guimarães | **RA:** 23110134 | **Turma:** 26-1-COMP-7-07-B

---

## Fundamentação Teórica

Treinar um modelo sem avaliá-lo corretamente é tão problemático quanto não treiná-lo. As principais métricas para classificação binária são:

| Métrica | Definição |
|---------|-----------|
| **Acurácia** | Proporção de predições corretas sobre o total |
| **Precisão** | Dos classificados como positivos, quantos realmente o são |
| **Recall** | Dos realmente positivos, quantos foram detectados |
| **F1-Score** | Média harmônica de precisão e recall |

As fórmulas são:

$$\text{Acurácia} = \frac{VP + VN}{VP + VN + FP + FN}$$

$$\text{Precisão} = \frac{VP}{VP + FP} \qquad \text{Recall} = \frac{VP}{VP + FN}$$

$$F_1 = 2 \cdot \frac{\text{Precisão} \times \text{Recall}}{\text{Precisão} + \text{Recall}}$$

A **separação treino/teste** é obrigatória — avaliar no mesmo conjunto usado para treinar sempre infla artificialmente as métricas. O **StandardScaler** deve ser ajustado somente nos dados de treino (`fit_transform`) e apenas aplicado no teste (`transform`), para evitar vazamento de informação.

In [1]:
# Passo 3 — Pipeline de avaliação completo
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)
from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
import numpy as np

# Gerar dataset sintético
X, y = make_classification(
    n_samples=1000,
    n_features=10,
    n_informative=5,
    n_redundant=2,
    random_state=42
)

# Dividir treino/teste
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Normalizar (fit no treino, transform nos dois)
scaler = StandardScaler()
X_treino = scaler.fit_transform(X_treino)
X_teste = scaler.transform(X_teste)

# Treinar modelo
modelo = MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=500, random_state=42)
modelo.fit(X_treino, y_treino)

y_pred = modelo.predict(X_teste)

print(f'Acurácia:  {accuracy_score(y_teste, y_pred):.4f}')
print(f'Precisão:  {precision_score(y_teste, y_pred):.4f}')
print(f'Recall:    {recall_score(y_teste, y_pred):.4f}')
print(f'F1-Score:  {f1_score(y_teste, y_pred):.4f}')
print()
print('Matriz de Confusão:')
print(confusion_matrix(y_teste, y_pred))
print()
print('Relatório Completo:')
print(classification_report(y_teste, y_pred))

Acurácia:  0.9400
Precisão:  0.9588
Recall:    0.9208
F1-Score:  0.9394

Matriz de Confusão:
[[95  4]
 [ 8 93]]

Relatório Completo:
              precision    recall  f1-score   support

           0       0.92      0.96      0.94        99
           1       0.96      0.92      0.94       101

    accuracy                           0.94       200
   macro avg       0.94      0.94      0.94       200
weighted avg       0.94      0.94      0.94       200



/private/var/www/facul/redes-neurais/trabalho/.venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


In [2]:
# Passo 4 — Detecção de overfitting e underfitting
from sklearn.neural_network import MLPClassifier
import numpy as np

# Modelo com overfitting (muito complexo)
modelo_over = MLPClassifier(hidden_layer_sizes=(128, 64, 32), max_iter=2000, random_state=42)
modelo_over.fit(X_treino, y_treino)
acc_over_treino = modelo_over.score(X_treino, y_treino)
acc_over_teste  = modelo_over.score(X_teste, y_teste)

# Modelo adequado
modelo_ok = MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=500, random_state=42)
modelo_ok.fit(X_treino, y_treino)
acc_ok_treino = modelo_ok.score(X_treino, y_treino)
acc_ok_teste  = modelo_ok.score(X_teste, y_teste)

# Modelo com underfitting (muito simples)
modelo_under = MLPClassifier(hidden_layer_sizes=(2,), max_iter=50, random_state=42)
modelo_under.fit(X_treino, y_treino)
acc_under_treino = modelo_under.score(X_treino, y_treino)
acc_under_teste  = modelo_under.score(X_teste, y_teste)

print('Comparação de modelos:')
print(f'{"Modelo":<15} {"Treino":>8} {"Teste":>8} {"Gap":>8}')
print('-' * 40)
print(f'{"Overfitting":<15} {acc_over_treino:>8.4f} {acc_over_teste:>8.4f} {acc_over_treino-acc_over_teste:>8.4f}')
print(f'{"Adequado":<15} {acc_ok_treino:>8.4f} {acc_ok_teste:>8.4f} {acc_ok_treino-acc_ok_teste:>8.4f}')
print(f'{"Underfitting":<15} {acc_under_treino:>8.4f} {acc_under_teste:>8.4f} {acc_under_treino-acc_under_teste:>8.4f}')
print()
print('Gap > 0.05: possível overfitting')
print('Treino < 0.75: possível underfitting')

Comparação de modelos:
Modelo            Treino    Teste      Gap
----------------------------------------
Overfitting       1.0000   0.9300   0.0700
Adequado          1.0000   0.9400   0.0600
Underfitting      0.6212   0.6000   0.0212

Gap > 0.05: possível overfitting
Treino < 0.75: possível underfitting


/private/var/www/facul/redes-neurais/trabalho/.venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/private/var/www/facul/redes-neurais/trabalho/.venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (50) reached and the optimization hasn't converged yet.
  warnings.warn(


In [3]:
# Passo 5 — Exercício: Dataset Diabetes
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
import numpy as np

# Carregar dados (regressão → binarizar pela mediana)
diabetes = load_diabetes()
X_diab = diabetes.data
y_diab = (diabetes.target > diabetes.target.mean()).astype(int)

X_tr, X_te, y_tr, y_te = train_test_split(
    X_diab, y_diab, test_size=0.2, random_state=42, stratify=y_diab
)

scaler_diab = StandardScaler()
X_tr = scaler_diab.fit_transform(X_tr)
X_te = scaler_diab.transform(X_te)

modelo_diab = MLPClassifier(
    hidden_layer_sizes=(16, 8),
    max_iter=500,
    random_state=42
)
modelo_diab.fit(X_tr, y_tr)

y_pred_diab = modelo_diab.predict(X_te)

print('Dataset Diabetes — Classificação (acima/abaixo da média):')
print(f'  Acurácia: {accuracy_score(y_te, y_pred_diab):.4f}')
print(f'  F1-Score: {f1_score(y_te, y_pred_diab):.4f}')
print()
print(classification_report(y_te, y_pred_diab))

Dataset Diabetes — Classificação (acima/abaixo da média):
  Acurácia: 0.7416
  F1-Score: 0.7089

              precision    recall  f1-score   support

           0       0.78      0.76      0.77        50
           1       0.70      0.72      0.71        39

    accuracy                           0.74        89
   macro avg       0.74      0.74      0.74        89
weighted avg       0.74      0.74      0.74        89



/private/var/www/facul/redes-neurais/trabalho/.venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


## Análise Crítica

**Métricas complementares:** A acurácia pode ser enganosa em datasets desbalanceados — um classificador que sempre prediz a classe majoritária pode ter acurácia alta. O F1-Score equilibra precisão e recall numa única métrica mais honesta, e a matriz de confusão revela exatamente onde o modelo erra.

**Overfitting:** O modelo de três camadas (128-64-32) apresenta gap expressivo entre treino e teste, sinal de memorização em vez de generalização. Técnicas como dropout ou regularização L2 mitigariam esse efeito.

**Underfitting:** O modelo com apenas 2 neurônios não tem capacidade para capturar os padrões do dataset — tanto treino quanto teste ficam com acurácia baixa.

**Vazamento de dados (data leakage):** Ajustar o scaler com os dados de teste é um erro clássico que infla as métricas artificialmente, dando uma falsa sensação de desempenho. O scaler deve sempre ser ajustado exclusivamente no conjunto de treino.